# Notebook 01 — Exploratory Data Analysis
**Prompt Injection & Jailbreak Detection System**

This notebook explores the dataset before training. Key goals:
- Understand class balance
- Check token length distribution
- Visualise label-discriminative word frequencies
- Compute class weights for weighted loss

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.utils.class_weight import compute_class_weight

BASE = os.getcwd()
DATA_DIR = os.path.join(BASE, "data")
os.makedirs(DATA_DIR, exist_ok=True)

def _pick_col(features, candidates):
    for c in candidates:
        if c in features:
            return c
    raise ValueError(f"None of {candidates} found in dataset features: {list(features)}")

raw = load_dataset("jayavibhav/prompt-injection")
rows = []
for split in raw.keys():
    text_col = _pick_col(raw[split].features, ["text", "prompt", "query"])
    label_col = _pick_col(raw[split].features, ["label", "target"])
    part = pd.DataFrame({
        "text": raw[split][text_col],
        "label": raw[split][label_col],
    })
    rows.append(part)

df = pd.concat(rows, ignore_index=True).dropna()
df["label"] = df["label"].astype(int)
df.head()

## 1. Load Dataset

In [ ]:
print(f"Shape: {df.shape}")
display(df.sample(5, random_state=42))
print("\nLabel counts:")
print(df["label"].value_counts().sort_index())

## 2. Label Distribution

In [ ]:
counts = df["label"].value_counts().sort_index()
label_names = ["SAFE (0)", "INJECTION (1)"]

plt.figure(figsize=(6, 4))
plt.bar(label_names, counts.values, color=["#27ae60", "#e74c3c"])
plt.title("Label Distribution")
plt.ylabel("Count")
plt.show()

print("Class balance:")
print(counts.to_dict())

## 3. Sample Queries

In [ ]:
print("Sample SAFE queries:\n")
for t in df[df["label"] == 0]["text"].head(5):
    print(f"- {str(t)[:140]}")

print("\nSample INJECTION queries:\n")
for t in df[df["label"] == 1]["text"].head(5):
    print(f"- {str(t)[:140]}")

## 4. Token Length Analysis
> We use this to confirm  is appropriate.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
df_tokens = df.copy()
df_tokens["token_len"] = df_tokens["text"].astype(str).apply(
    lambda x: len(tokenizer.encode(x, truncation=False))
)

plt.figure(figsize=(8, 4))
for label, color, name in [(0, "#27ae60", "SAFE"), (1, "#e74c3c", "INJECTION")]:
    subset = df_tokens[df_tokens["label"] == label]["token_len"]
    plt.hist(subset, bins=40, alpha=0.6, color=color, label=f"{name} (mean={subset.mean():.0f})")
plt.axvline(128, color="navy", linestyle="--", label="max_len=128")
plt.title("Token Length Distribution")
plt.xlabel("Token count")
plt.ylabel("Frequency")
plt.legend()
plt.show()

print(df_tokens["token_len"].describe())

## 5. Top Words per Class

In [ ]:
import re
from collections import Counter

def simple_tokens(text):
    return re.findall(r"[a-zA-Z']+", str(text).lower())

stop = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "with",
    "is", "are", "be", "as", "you", "your", "i", "me", "my", "we", "our",
    "this", "that", "it", "from", "at", "by", "can", "please", "what", "how"
}

safe_counter = Counter()
inj_counter = Counter()

for t in df[df["label"] == 0]["text"]:
    safe_counter.update([w for w in simple_tokens(t) if w not in stop and len(w) > 2])
for t in df[df["label"] == 1]["text"]:
    inj_counter.update([w for w in simple_tokens(t) if w not in stop and len(w) > 2])

top_safe = safe_counter.most_common(15)
top_inj = inj_counter.most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh([w for w, _ in top_safe][::-1], [c for _, c in top_safe][::-1], color="#27ae60")
axes[0].set_title("Top Words in SAFE")
axes[1].barh([w for w, _ in top_inj][::-1], [c for _, c in top_inj][::-1], color="#e74c3c")
axes[1].set_title("Top Words in INJECTION")
plt.tight_layout()
plt.show()

## 6. Class Weights for Weighted Loss

In [ ]:
classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["label"].values,
 )
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weights)